# Functions — Banking Mini Project

**Class 9 notes.** A complete ATM-style banking system built with functions:
every operation (open, balance, withdraw, deposit, transfer, close) is its own function that
works on a shared list of account dictionaries — functions in action on real data.

**What you will learn**
- Defining functions with `def`, parameters and default values
- Storing records in a `list` of `dict`s
- Searching with `for` loops (and nested loops for transfers)
- `for / else`, `enumerate()` and deleting while iterating

## 1. Data model & open_account()

- Storage: `all_accounts = []` — a **list of dictionaries**, one dict per account.
- `open_account(cnic, pin, deposit=0)` builds a dict with a random `account_id`, the given
  CNIC/PIN and the opening balance, then appends it to the list.

In [ ]:
# open_account(): default parameter deposit=0 lets you open an account without a deposit
import random
all_accounts = []

userNic=5544
userPin=1122
beneficiary_acc = 0

def open_account(cnic, pin, deposit=0):
    account_id = random.randint(1000, 9999)
    account_title = input("Enter account title:")
    account_number = cnic
    account_pin = pin
    account = {
        "account_id": account_id,
        "account_title": account_title,
        "account_number": account_number,
        "account_pin": account_pin,
        "account_balance": deposit,
    }
    all_accounts.append(account)
    return account

open_account(userNic, userPin, 5000)

## 2. check_balance()

- Loops through every account and matches `account_number` **and** `account_pin`.
- The `else:` belongs to the **`for` loop** (not the `if`): it runs only when the loop
  finishes without returning — i.e. the account number was never found.

In [ ]:
# for/else: the else runs only when the loop never returned (account not found)
def check_balance(user_account, user_pin):
    for acc in all_accounts:
        if acc["account_number"] == user_account:
            if acc["account_pin"] == user_pin:
                return acc["account_balance"]
            else:
                return "Invalid Pin"

    else:
        return f"{user_account} is invalid account"

check_balance(userNic, userPin)

## 3. cash_withdrawl()

- Same search pattern, plus a balance check: `if acc["account_balance"] >= amount:`.
- The balance is updated **inside the dictionary** (`-=`), so the change persists.

In [ ]:
# withdraw only happens after BOTH the PIN and the balance check pass
def cash_withdrawl(user_account, user_pin, amount):
    for acc in all_accounts:
        if acc["account_number"] == user_account:
            if acc["account_pin"] == user_pin:
                if acc["account_balance"] >= amount:
                    acc["account_balance"] -= amount
                    return f"{amount} withdrawl succesfully"
                else:
                    return "Insufficient Balance!"
            else:
                return "Invalid Pin"
    else:
        return f"{user_account} is invalid account"

cash_withdrawl(userNic, userPin, 1000)

## 4. cash_deposit()

- Simplest flow: find the account → `acc["account_balance"] += amount` → confirm.

In [ ]:
# deposit finds the account, then updates its balance in place
def cash_deposit(user_account, amount):
    for acc in all_accounts:
        if acc["account_number"] == user_account:
            acc["account_balance"] += amount
            return f"{amount} deposit successfully!"
    else:
        return f"{user_account} is invalid account"

cash_deposit(userNic, 1000)

## 5. transfer_amount()

- **Nested loops:** the outer loop finds the sender, the inner loop finds the beneficiary.
- Both PIN and balance are validated on the sender before any money moves.

In [ ]:
# transfer: outer loop = sender, inner loop = beneficiary
def transfer_amount(user_account, user_pin, amount, beneficiary_account):
    for acc in all_accounts:
        if acc["account_number"] == user_account:
            if acc["account_pin"] == user_pin:
                if acc["account_balance"] >= amount:
                    for bacc in all_accounts:
                        if bacc["account_number"] == beneficiary_account:
                            acc["account_balance"] -= amount
                            bacc["account_balance"] += amount
                            return f"{amount} transfered successfully"
                    else:
                        return f"{beneficiary_account} is invalid account"
                else:
                    return "Insufficient Balance!"
            else:
                return "Invalid Pin"
    else:
        return f"{user_account} is invalid account"

transfer_amount(userNic, userPin, 50000, bacc)

## 6. close_account()

- `enumerate()` gives the index **and** the account, so `del all_accounts[i]` can remove it.
- The final balance is returned to the user before the account is deleted.

In [ ]:
# enumerate() gives (index, account) so we can delete by index
def close_account(user_account, user_pin):
    for i, acc in enumerate(all_accounts):
        if acc["account_number"] == user_account:
            if acc["account_pin"] == user_pin:
                remaining_amount = acc["account_balance"]
                acc["account_balance"] = 0
                del all_accounts[i]
                return f"""Collect your amount {remaining_amount} {acc['account_number']} closed successfully!"""

    else:
        return f"{user_account} is invalid account"

close_account(userNic, userPin)

## 7. Looping tricks

- Removing an item **while looping** over the same list skips the next item —
  Python moves the index forward even though the list shrank.
- Safer alternatives: iterate over a copy (`for x in lst[:]:`) or rebuild the list.

In [ ]:
# classic gotcha: removing while iterating skips the element after the removed one
lst = ["a", "d", "r", "t"]
for name in lst:
    if name == "r":
        print("found r — removing it")
        lst.remove(name)   # list shrinks while looping: "t" is skipped!
lst

In [ ]:
# enumerate() also lets us print the index of every account while looping
for i, acc in enumerate(all_accounts):
    print(i, acc)

## 🔑 Key Takeaways / Quick Revision

- `def name(params):` defines a function; call it with `name(args)` — `deposit=0` gives a default value.
- All accounts live in `all_accounts = []` as dictionaries: `{"account_id": ..., "account_balance": ...}`.
- Searching = loop over the list and compare `acc["account_number"] == user_account`.
- `return` sends a value back to the caller — it also **exits** the function immediately.
- `for ... else:` → the `else` block runs when the loop finishes without hitting a `break`
  (used here for the "account not found" message).
- `transfer_amount()` uses a **nested loop**: outer = sender, inner = beneficiary.
- `enumerate(lst)` gives `(index, item)` pairs — handy for deleting by index inside the loop.
- ⚠️ Removing items **while looping over the same list** skips the next element — prefer
  building a new list or iterating over a copy.